# Polygons to polylines (Fiji)

- This script expects a RoiManager full of polygon rois and an opened image.
- Each polygon in the manager will be skeletonized and turned into a polyline.
- At the end, the manager is flushed and filled with the polylines.

```python
from ij import IJ, ImagePlus
from ij.process import ByteProcessor
from ij.gui import PolygonRoi, Roi
from ij.plugin.frame import RoiManager
from java.util import LinkedList


def get_neighbors(pixels, x, y, width, height):
    neighbors = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < width and 0 <= ny < height:
                if pixels[ny * width + nx] != 0:
                    neighbors.append((nx, ny))
    return neighbors

def get_all_skeleton_pixels(pixels, width, height):
    return [(x, y) for y in range(height) for x in range(width)
            if pixels[y * width + x] != 0]

def find_endpoints_and_junctions(pixels, width, height):
    endpoints = []
    junctions = []
    for x, y in get_all_skeleton_pixels(pixels, width, height):
        n = len(get_neighbors(pixels, x, y, width, height))
        if n == 1:
            endpoints.append((x, y))
        elif n >= 3:
            junctions.append((x, y))
    return endpoints, junctions

def trace_branch(pixels, width, height, start, came_from):
    path = [start]
    prev = came_from
    current = start
    while True:
        neighbors = [n for n in get_neighbors(pixels, current[0], current[1], width, height)
                     if n != prev]
        if len(neighbors) != 1:
            break
        prev = current
        current = neighbors[0]
        path.append(current)
    return path

def remove_small_branches(pixels, width, height, min_branch_length):
    changed = True
    while changed:
        changed = False
        endpoints, junctions = find_endpoints_and_junctions(pixels, width, height)
        junction_set = set(junctions)
        for ep in endpoints:
            branch = trace_branch(pixels, width, height, ep, None)
            tip = branch[-1]
            if tip in junction_set and len(branch) < min_branch_length:
                for px, py in branch:
                    pixels[py * width + px] = 0
                changed = True

def find_longest_path(pixels, width, height):
    endpoints, _ = find_endpoints_and_junctions(pixels, width, height)
    if len(endpoints) < 2:
        IJ.log("Warning: expected 2 endpoints, found " + str(len(endpoints)))
        if len(endpoints) == 0:
            return []
        all_px = get_all_skeleton_pixels(pixels, width, height)
        endpoints = [all_px[0], all_px[len(all_px) // 2]]

    start = endpoints[0]
    visited = {}
    queue = LinkedList()
    queue.add(start)
    visited[start] = None
    last = start

    while not queue.isEmpty():
        current = queue.poll()
        last = current
        for nb in get_neighbors(pixels, current[0], current[1], width, height):
            if nb not in visited:
                visited[nb] = current
                queue.add(nb)

    path = []
    node = last
    while node is not None:
        path.append(node)
        node = visited[node]
    path.reverse()
    return path

def smooth_and_subsample_path(path, window=5, step=5):
    if len(path) < window:
        return path
    smoothed = []
    half = window // 2
    for i in range(len(path)):
        lo = max(0, i - half)
        hi = min(len(path), i + half + 1)
        xs = [p[0] for p in path[lo:hi]]
        ys = [p[1] for p in path[lo:hi]]
        smoothed.append((sum(xs) / len(xs), sum(ys) / len(ys)))
    subsampled = [smoothed[i] for i in range(0, len(smoothed), step)]
    if subsampled[-1] != smoothed[-1]:
        subsampled.append(smoothed[-1])
    return subsampled

def skeletonize_imp(imp):
    """Run the proper Skeletonize (2D/3D) plugin on an ImagePlus in place."""
    IJ.run(imp, "Skeletonize (2D/3D)", "")

def skeleton_to_polyline(pixels, width, height, min_branch_length=20):
    remove_small_branches(pixels, width, height, min_branch_length)
    path = find_longest_path(pixels, width, height)
    if len(path) < 2:
        return None
    path = smooth_and_subsample_path(path, window=5, step=5)
    xs = [int(round(p[0])) for p in path]
    ys = [int(round(p[1])) for p in path]
    return PolygonRoi(xs, ys, len(path), Roi.POLYLINE)


# ── Main ──────────────────────────────────────────────────────────────────────

imp = IJ.getImage()
width  = imp.getWidth()
height = imp.getHeight()

rm = RoiManager.getInstance()
if rm is None:
    rm = RoiManager()

roi_count = rm.getCount()
source_rois = [rm.getRoi(i) for i in range(roi_count)]

IJ.log("Processing " + str(roi_count) + " ROIs...")

polyline_rois = []

for idx, roi in enumerate(source_rois):
    IJ.log("  ROI " + str(idx + 1) + "/" + str(roi_count) + " ...")

    # Create a blank 8-bit ImagePlus and fill the ROI with white
    bp = ByteProcessor(width, height)
    bp.setValue(0)
    bp.fill()
    bp.setValue(255)
    bp.fill(roi)

    tmp = ImagePlus("skel_tmp_" + str(idx), bp)

    # Run the proper Skeletonize (2D/3D) plugin
    skeletonize_imp(tmp)

    # Debug: count non-zero pixels after skeletonization
    java_pixels = bp.getPixels()
    pixels = [java_pixels[i] & 0xff for i in range(width * height)]
    nonzero = sum(1 for v in pixels if v != 0)
    IJ.log("    non-zero pixels after skeletonize: " + str(nonzero))

    result_roi = skeleton_to_polyline(pixels, width, height)

    if result_roi is not None:
        polyline_rois.append(result_roi)
        IJ.log("    -> polyline with " + str(result_roi.getNCoordinates()) + " points")
    else:
        IJ.log("    -> skipped (no valid path found)")

    tmp.close()

# Reset the ROI manager and repopulate with polylines
rm.reset()
for r in polyline_rois:
    rm.addRoi(r)

rm.runCommand("Show All")
IJ.log("Done. " + str(len(polyline_rois)) + " polyline ROIs added to ROI manager.")
```